(Tutorial_Get_pairwise_rmsd)=
# Getting pairwise pose RMSD

*Comparing a bounded population without changing its position or orientation.*

Use this experimental tool when all structures share a declared atom axis and
Cartesian reference. It does not align, center or reconstruct periodic images.
A coordinate-only form can use `selection='all'`; the default heavy-atom selection
requires atom inventory. Input may be any form supplying the required attributes.

:::{admonition} API documentation
:class: dropdown
{func}`molsysmt.structure.get_pairwise_rmsd`
:::

:::{versionadded} 1.0.0
:::

## Preserving member identity

These analytic structures differ only by translation. Repeated structure indices
remain separate members, with unique caller labels. Numeric atom selections are
unique and reported in sorted original source positions. Indices are not IDs.


In [1]:
import numpy as np
import molsysmt as msm
from molsysmt.native import Structures
puw = msm.pyunitwizard
coordinates = np.zeros((3, 2, 3))
coordinates[:, :, 0] = np.array([0., 1., 2.])[:, None]
molsys = Structures(coordinates=puw.quantity(coordinates, 'nm'))
result = msm.structure.get_pairwise_rmsd(
    molsys, selection='all', structure_indices=[2, 0, 2],
    member_keys=['pose-c', 'pose-a', 'pose-c-again'])
print(result['structure_indices'].tolist())
print(puw.get_value(result['rmsd'], to_unit='nm'))
assert np.allclose(puw.get_value(result['rmsd'], to_unit='nm'),
                   [[0, 2, 0], [2, 0, 2], [0, 2, 0]])


[2, 0, 2]
[[0. 2. 0.]
 [2. 0. 2.]
 [0. 2. 0.]]


## Declaring chemical symmetry

`method='positional'` uses the declared atom order and checks no chemistry.
`method='symmetry_corrected'` reuses the declared fixed-state profile of
{func}`molsysmt.chemical_states.get_atom_correspondences` and requires its complete
chemical context. You can pass a reusable `correspondences` dictionary.
Its maps must refer to this same selected atom population and chemical state on
both axes, contain identity and include the inverse of every map. Directional
candidate sets are rejected, never averaged. Keep `search_scope`: a restricted
candidate/anchored set need not define a metric or the unrestricted minimum.

`pair_indices` rows are `(query_member, reference_member)` with query > reference,
ordered by reference then query. Map winners and near-tie counts belong to that
orientation only. The matrix mirrors the value after symmetry admission; it does
not invent a reverse winning map. The shared map table is stored only once.

Separate saved systems need explicit mapping and preparation before composition.
`concatenate_structures` does not authenticate chemistry or common coordinates.
See {ref}`Cookbook_ClusteringPoses`.

## Bounding resources and retaining units

`max_work` bounds unordered pairs × selected atoms × maps. Configured RAM checks
bound selected numeric coordinates, matrix/witness outputs and map working
estimates. Source/Python chemistry objects and total process RSS are excluded.
Coordinate input may stream, but selected coordinates and dense output remain
resident. This is a bounded population analysis, not an out-of-core trajectory
matrix. Empty requests have shape `(0, 0)` and evaluate no coordinates or chemistry;
singleton coordinates are validated before returning zero. Nonfinite coordinates,
unsupported chemistry, incomplete maps and budget violations raise clearly.


In [2]:
with puw.context(standard_units=['angstrom', 'ps']):
    converted = msm.structure.get_pairwise_rmsd(molsys, selection='all')
    print(puw.get_value(converted['rmsd']))
    assert np.allclose(puw.get_value(converted['rmsd'])[0], [0, 10, 20])


[[ 0. 10. 20.]
 [10.  0. 10.]
 [20. 10.  0.]]


:::{seealso}
{ref}`Tutorial_Get_distance_clusters`, {ref}`Tutorial_Get_symmetry_corrected_rmsd`
and {ref}`Cookbook_ClusteringPoses` explain clustering, chemical admission and
preserving original member keys. Detached bibliography is present independently
of optional Ackredit tracking. Source coordinates are never changed.
:::
